# Lesson 04 - Tool Use Design Pattern

For dis lesson, you go learn the **Tool Use** design pattern for AI agents wey dey use Microsoft Agent Framework (Python). We go cover:

- How to define function tools wit de `@tool` decorator and typed parameters
- How to provide tool schemas so dat de model sabi wetin each tool dey do
- How to control tool execution wit `approval_mode`
- How to return **structured output** via Pydantic models and `response_format`

The scenario na **travel booking agent** wey fit look up destinations, check availability, and retrieve flight information.


## Setup


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Defining Tools with the @tool Decorator

The `@tool` decorator dey turn one plain Python function to tool wey agent fit call.
Key points:

- The **docstring** na im be tool description wey model go see.
- **Type annotations** (including `Annotated` wey get descriptions) dey define tool schema.
- `approval_mode` dey control whether user must approve each call before e run.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Creating an Agent wit Multiple Tools

Pass all three tools give di client so di model fit use any one wey e need to answer di user's question.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Structured Output wit Tools

By setting `response_format` to a Pydantic model, di agent go forced to return one well-typed JSON object instead of free-form text. Dis one dey useful wen downstream code need to consume di result programmatically.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Tool Approval Patterns

Di `approval_mode` parameter wey dey for `@tool` dey control whether tool calls go need human approval before e fit run:

| Mode | Behaviour |
|---|---|
| `"never_require"` | Tool go run automatically — no need for user confirmation. |
| `"always_require"` | Every call must get user approval before e fit run. |

Use `"always_require"` for tools wey get side-effects (like booking flight, charging credit card) so person go dey involved.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Summary

For dis lesson, you learn how to:

1. **Define tools** using di `@tool` decorator wit typed parameters and docstrings wey dey serve as di tool schema.
2. **Compose multiple tools** make di agent fit call dem one by one to answer complex questions.
3. **Return structured output** by passing Pydantic model as `response_format`.
4. **Control tool approval** wit `approval_mode` to keep human dey involved for sensitive operations.

Dem patterns na di foundation for building reliable, production-ready agents wey fit interact wit external systems safely.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:  
Dis document don translate wit AI translation service wey dem call [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg sabi say machine translation fit get some mistakes or no too correct. Di original document for e own language na di main source wey get correct info. If na serious matter, better make person wey sabi human translation do am. We no go responsible if pesin comot wrong understanding from dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
